# 02. Order Base Validation

## Objective

Construct and validate a one-row-per-order analytical base before creating the first-purchase cohort and 90-day outcome.

The validation focuses on:

- order-level grain
- aggregation consistency
- payment-approved purchase definition
- prediction snapshot
- observation-period boundary

In [1]:
import sqlite3

import pandas as pd

In [2]:
conn = sqlite3.connect(
    "../data/processed/olist.db"
)

order_base_df = pd.read_sql_query(
    """
    SELECT *
    FROM order_base
    """,
    conn
)

conn.close()

In [3]:
# Order Grain 확인
print(
    "rows:",
    len(order_base_df)
)

print(
    "unique order_id:",
    order_base_df["order_id"]
    .nunique()
)

print(
    "duplicate order_id:",
    order_base_df["order_id"]
    .duplicated()
    .sum()
)

rows: 99441
unique order_id: 99441
duplicate order_id: 0


### Finding — Order-Level Grain

The order base contains one row per order.

No duplicate order_id remains after item and payment tables are aggregated before the joins.

This structure prevents one-to-many joins from inflating transaction values.

In [4]:
# Snapshot 확인
approved_order_df = (
    order_base_df[
        order_base_df[
            "is_approved_order"
        ] == 1
    ]
)

print(
    approved_order_df[
        "order_status"
    ]
    .value_counts()
)

order_status
delivered      96464
shipped         1107
unavailable      609
canceled         484
invoiced         314
processing       301
approved           2
Name: count, dtype: int64


### Decision — Prediction Snapshot

The primary prediction snapshot is payment approval(order_approved_at).

Orders without a recorded approval timestamp are excluded from the primary purchase definition.

Final order status is retained for audit purposes, but is not used as a model feature because it may only be known after the prediction snapshot.